<a href="https://colab.research.google.com/github/chaunijs/onlineshoppingprice/blob/main/big_C_cloudflare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
import subprocess
import sys
from IPython.display import display, HTML

# 1. Setup the 2-line scrolling box UI
display(HTML("""
    <style>
        #scroll_output {
            height: 50px; /* Approximately 2 lines */
            overflow-y: scroll;
            background-color: #1e1e1e;
            color: #00ff00;
            padding: 10px;
            font-family: monospace;
            font-size: 12px;
            border: 1px solid #444;
            display: flex;
            flex-direction: column;
        }
    </style>
    <div id="scroll_output">Starting installation...</div>
"""))

def run_and_scroll(commands):
    """Runs list of commands and streams output to the scroll_output div"""
    for cmd in commands:
        process = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            # Escape single quotes for JS and update the div
            escaped_line = line.replace("'", "\\'").strip()
            display(HTML(f"""<script>
                var obj = document.getElementById('scroll_output');
                obj.innerHTML += '<div>' + '{escaped_line}' + '</div>';
                obj.scrollTop = obj.scrollHeight;
            </script>"""), display_id='scroll_update')
        process.wait()

# 2. Updated list of commands (Includes OCR + Scraping tools)
commands_to_run = [
    "apt-get update",
    # Install system-level chrome
    "apt-get install -y google-chrome-stable",
    # Install all Python libraries in one go
    "pip install easyocr pillow requests polars scrapling patchright msgspec browserforge nest_asyncio -q",
    # Install the patched browser and its system dependencies
    "patchright install chromium",
    "patchright install-deps"
]

run_and_scroll(commands_to_run)
print("\n✅ All installations finished.")


✅ All installations finished.


In [2]:
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import polars as pl
import asyncio
import xlsxwriter
import datetime
from datetime import date
import IPython.display as display
import datetime
today_date = datetime.datetime.now().strftime("%Y-%m-%d")
print("Today is",today_date)

Today is 2026-04-28


In [3]:
%%script echo skipping
# @title work
# This code works
import asyncio
import nest_asyncio
import pandas as pd
from scrapling.fetchers import StealthyFetcher

nest_asyncio.apply()

async def scrape_bigc_multi_pages():
    base_url = "https://www.bigc.co.th/en/category/laundry?brand=184%2C249%2C188%2C189%2C185"
    current_page = 2
    all_data = []

    # กำหนดค่าภาษาอังกฤษ
    en_cookies = [
        {'name': 'language', 'value': 'en', 'domain': '.bigc.co.th', 'path': '/'},
        {'name': 'NEXT_LOCALE', 'value': 'en', 'domain': '.bigc.co.th', 'path': '/'}
    ]
    en_headers = {'Accept-Language': 'en-US,en;q=0.9'}

    print("เริ่มการดึงข้อมูลหลายหน้า...")

    while True:
        target_url = f"{base_url}&page={current_page}"
        print(f"กำลังดึงข้อมูลหน้า {current_page}: {target_url}")

        try:
            page = await StealthyFetcher.async_fetch(
                target_url,
                headless=True,
                solve_cloudflare=True,
                cookies=en_cookies,
                headers=en_headers,
                timeout=60000,
                network_idle=True
            )

            if page.status == 200:
                containers = page.css('div[class*="productCard_container"]')

                # ถ้าไม่พบสินค้าในหน้านี้ แสดงว่าถึงหน้าสุดท้ายแล้ว
                if not containers:
                    print(f"ไม่พบสินค้าเพิ่มเติมที่หน้า {current_page} จบการทำงาน")
                    break

                print(f"พบสินค้า {len(containers)} รายการในหน้า {current_page}")

                for item in containers:
                    name = item.css('div[class*="productCard_title"] a::text').get()
                    sale_price = item.css('span[class*="productCard_sale_price"]::text').get()
                    original_price = item.css('div[class*="productCard_base_price"]::text').get()

                    all_data.append({
                        "page": current_page,
                        "product_name": name.strip() if name else "N/A",
                        "sale_price": sale_price.strip() if sale_price else "N/A",
                        "original_price": original_price.strip().replace('฿', '') if original_price else "N/A"
                    })

                # เพิ่มเลขหน้าเพื่อไปหน้าถัดไป
                current_page += 1

                # ใส่หน่วงเวลาเล็กน้อยเพื่อไม่ให้โดนบล็อก
                await asyncio.sleep(2)
            else:
                print(f"เกิดข้อผิดพลาดที่หน้า {current_page} (Status: {page.status}) หยุดการทำงาน")
                break

        except Exception as e:
            print(f"เกิดข้อผิดพลาดที่หน้า {current_page}: {e}")
            break

    # สรุปผลลัพธ์
    if all_data:
        df = pd.DataFrame(all_data)
        print(f"\nดึงข้อมูลสำเร็จทั้งหมด {len(df)} รายการ จากหน้า 2 ถึง {current_page-1}")
        display(df)
        df.to_csv('bigc_laundry_full.csv', index=False)
        print("บันทึกข้อมูลลงไฟล์ bigc_laundry_full.csv เรียบร้อยแล้ว")
    else:
        print("ไม่พบข้อมูลที่จะบันทึก")

# เริ่มทำงาน
await scrape_bigc_multi_pages()

skipping


In [4]:
# @title UDF scrape big C polars
# 1. ติดตั้งไลบรารีที่จำเป็น (รวมถึง polars)
# !pip install scrapling patchright msgspec browserforge nest_asyncio polars -q
# !patchright install chromium
# !patchright install-deps

import asyncio
import nest_asyncio
import polars as pl
from scrapling.fetchers import StealthyFetcher

nest_asyncio.apply()

async def get_scrape_bigc_multi_pages_polars(base_url):
    current_page = 1
    all_data = []

    # กำหนดค่าคุกกี้และ headers เพื่อรักษาเซสชันภาษาอังกฤษ
    en_cookies = [
        {'name': 'language', 'value': 'en', 'domain': '.bigc.co.th', 'path': '/'},
        {'name': 'NEXT_LOCALE', 'value': 'en', 'domain': '.bigc.co.th', 'path': '/'}
    ]
    en_headers = {'Accept-Language': 'en-US,en;q=0.9'}

    print("เริ่มการดึงข้อมูลหลายหน้าด้วย Polars...")

    while True:
        # ตรวจสอบโครงสร้าง URL เพื่อต่อพารามิเตอร์ page ให้ถูกต้อง
        separator = "&" if "?" in base_url else "?"
        target_url = f"{base_url}{separator}page={current_page}"
        print(f"กำลังดึงข้อมูลหน้า {current_page}: {target_url}")

        try:
            # ใช้ StealthyFetcher เพื่อข้าม Cloudflare WAF
            page = await StealthyFetcher.async_fetch(
                target_url,
                headless=True,
                solve_cloudflare=True, # จัดการกับระบบตรวจสอบบอทอัตโนมัติ
                cookies=en_cookies,
                headers=en_headers,
                timeout=60000,
                network_idle=True
            )

            if page.status == 200:
                containers = page.css('div[class*="productCard_container"]')

                # หากไม่พบสินค้าในหน้านี้ แสดงว่าถึงหน้าสุดท้ายแล้ว
                if not containers:
                    print(f"ไม่พบสินค้าเพิ่มเติมที่หน้า {current_page} จบการทำงาน")
                    break

                print(f"พบสินค้า {len(containers)} รายการในหน้า {current_page}")

                for item in containers:
                    # สกัดข้อมูลชื่อสินค้าและราคาจาก DOM
                    name = item.css('div[class*="productCard_title"] a::text').get()
                    sale_price = item.css('span[class*="productCard_sale_price"]::text').get()
                    original_price = item.css('div[class*="productCard_base_price"]::text').get()

                    # # Extract the availability status/badge
                    # status = item.css('div[class*="productCard_badge"]::text').get()
                    # if not status:
                    #     status = item.css('div[class*="productCard_label"]::text').get()

                    # # Map "Available soon" or "Out of stock"
                    # is_available = True
                    # if status and ("Soon" in status or "หมด" in status):
                    #     is_available = False

                    # Check for the "Available soon" or "Sold out" badge
                    # Usually found in productCard_badge or productCard_label
                    badge_text = item.css('div[class*="productCard_badge"]::text').get() or ""

                    all_data.append({
                        "page": current_page,
                        "product_name": name.strip() if name else "N/A",
                        "sale_price": sale_price.strip() if sale_price else "N/A",
                        "original_price": original_price.strip().replace('฿', '') if original_price else "N/A",
                        "status": badge_text.strip() if badge_text else "Available"
                    })

                current_page += 1
                # หน่วงเวลาเล็กน้อยเพื่อหลีกเลี่ยงการถูกตรวจจับพฤติกรรม
                await asyncio.sleep(2)
            else:
                print(f"เกิดข้อผิดพลาดที่หน้า {current_page} (Status: {page.status}) หยุดการทำงาน")
                break

        except Exception as e:
            print(f"เกิดข้อผิดพลาดที่หน้า {current_page}: {e}")
            break

    # สรุปผลลัพธ์และส่งคืนค่าเป็น Polars DataFrame
    if all_data:
        df = pl.DataFrame(all_data)
        print(f"\nดึงข้อมูลสำเร็จทั้งหมด {len(df)} รายการ จากหน้า 1 ถึง {current_page-1}")
        return df
    else:
        print("ไม่พบข้อมูลที่จะประมวลผล")
        return pl.DataFrame()

In [33]:
import asyncio
import nest_asyncio
import polars as pl
from scrapling.fetchers import StealthyFetcher

nest_asyncio.apply()

async def scrape_bigc_multi_pages(base_url_list: list):
    all_data = []

    en_cookies = [
        {'name': 'language', 'value': 'en', 'domain': '.bigc.co.th', 'path': '/'},
        {'name': 'NEXT_LOCALE', 'value': 'en', 'domain': '.bigc.co.th', 'path': '/'}
    ]
    en_headers = {'Accept-Language': 'en-US,en;q=0.9'}

    print(f"🚀 Starting Multi-Category Scrape ({len(base_url_list)} categories)...")

    for base_url in base_url_list:
        current_page = 1
        print(f"\n📂 Processing Category: {base_url}")

        while True:
            separator = "&" if "?" in base_url else "?"
            # Ensure we don't stack multiple page params
            clean_url = base_url.split(f"{separator}page=")[0]
            target_url = f"{clean_url}{separator}page={current_page}"

            print(f"   📄 Page {current_page}...")

            try:
                page = await StealthyFetcher.async_fetch(
                    target_url,
                    headless=True,
                    solve_cloudflare=True,
                    cookies=en_cookies,
                    headers=en_headers,
                    timeout=60000,
                    network_idle=True
                )

                if page.status != 200:
                    print(f"   🛑 Stopped at page {current_page} (Status: {page.status})")
                    break

                containers = page.css('div[class*="productCard_container"]')
                if not containers:
                    print(f"   ✅ Category Complete.")
                    break

                for item in containers:
                    name = item.css('div[class*="productCard_title"] a::text').get()
                    sale_price = item.css('span[class*="productCard_sale_price"]::text').get()
                    original_price = item.css('div[class*="productCard_base_price"]::text').get()
                    badge_url = item.css('div[class*="productCard_badge"] img::attr(src)').get()

                    all_data.append({
                        "product_name": name.strip() if name else "N/A",
                        "sale_price": sale_price.strip() if sale_price else "N/A",
                        "original_price": original_price.strip().replace('฿', '') if original_price else "N/A",
                        "condition": None, # Placeholder for Part 2
                        "badge_url": badge_url if badge_url else "null"
                    })

                current_page += 1
                await asyncio.sleep(1) # Be polite to the server

            except Exception as e:
                print(f"   ❌ Error at page {current_page}: {e}")
                break

    return pl.DataFrame(all_data) if all_data else pl.DataFrame()

# --- RUN SCRAPER ---
category_list = [
    "https://www.bigc.co.th/en/category/laundry?brand=187%2C188%2C249%2C186%2C256&limit=100",
    "https://www.bigc.co.th/en/category/household-dishwashing?limit=100"
]

final_df = await scrape_bigc_multi_pages(category_list)
print(f"🏁 Scrape finished. Total products: {len(final_df)}")

🚀 Starting Multi-Category Scrape (2 categories)...

📂 Processing Category: https://www.bigc.co.th/en/category/laundry?brand=187%2C188%2C249%2C186%2C256&limit=100
   📄 Page 1...


[2026-04-28 08:03:18] ERROR: No Cloudflare challenge found.
ERROR:scrapling:No Cloudflare challenge found.
[2026-04-28 08:04:18] INFO: Fetched (200) <GET https://www.bigc.co.th/en/category/laundry?brand=187%2C188%2C249%2C186%2C256&limit=100&page=1> (referer: https://www.google.com/)
INFO:scrapling:Fetched (200) <GET https://www.bigc.co.th/en/category/laundry?brand=187%2C188%2C249%2C186%2C256&limit=100&page=1> (referer: https://www.google.com/)


   📄 Page 2...


[2026-04-28 08:05:31] ERROR: No Cloudflare challenge found.
ERROR:scrapling:No Cloudflare challenge found.
[2026-04-28 08:06:32] INFO: Fetched (200) <GET https://www.bigc.co.th/en/category/laundry?brand=187%2C188%2C249%2C186%2C256&limit=100&page=2> (referer: https://www.google.com/)
INFO:scrapling:Fetched (200) <GET https://www.bigc.co.th/en/category/laundry?brand=187%2C188%2C249%2C186%2C256&limit=100&page=2> (referer: https://www.google.com/)


   📄 Page 3...


...

   📄 Page 5...


[2026-04-28 08:09:06] ERROR: No Cloudflare challenge found.
ERROR:scrapling:No Cloudflare challenge found.
[2026-04-28 08:09:06] INFO: Fetched (200) <GET https://www.bigc.co.th/en/category/laundry?brand=187%2C188%2C249%2C186%2C256&limit=100&page=5> (referer: https://www.google.com/)
INFO:scrapling:Fetched (200) <GET https://www.bigc.co.th/en/category/laundry?brand=187%2C188%2C249%2C186%2C256&limit=100&page=5> (referer: https://www.google.com/)


   ✅ Category Complete.

📂 Processing Category: https://www.bigc.co.th/en/category/household-dishwashing?limit=100
   📄 Page 1...


[2026-04-28 08:09:13] ERROR: No Cloudflare challenge found.
ERROR:scrapling:No Cloudflare challenge found.
[2026-04-28 08:09:14] INFO: Fetched (307) <GET https://www.bigc.co.th/en/category/household-dishwashing?limit=100&page=1> (referer: https://www.google.com/)
INFO:scrapling:Fetched (307) <GET https://www.bigc.co.th/en/category/household-dishwashing?limit=100&page=1> (referer: https://www.google.com/)
[2026-04-28 08:09:14] INFO: Fetched (200) <GET https://www.bigc.co.th/> (referer: https://www.google.com/)
INFO:scrapling:Fetched (200) <GET https://www.bigc.co.th/> (referer: https://www.google.com/)


   ✅ Category Complete.
🏁 Scrape finished. Total products: 332


In [35]:
# @title Processing OCR
import io
import re
import requests
import easyocr
import polars as pl
import numpy as np
from PIL import Image

# 1. Initialize OCR
print("🔄 Initializing OCR Engine...")
reader = easyocr.Reader(['en', 'th'])

def get_condition_from_text(raw_text):
    """
    Handles dynamic patterns: Buy 2/3/4/5 Get 1 and Buy 2/3/4 Cheaper.
    Also normalizes Thai keywords and common OCR typos.
    """
    # Normalize text
    t = raw_text.upper().replace(" ", "")
    t = t.replace("BUV", "BUY")  # Fix common OCR typo

    # Extract the first digit found in the text (e.g., '3' from 'BUY3GET1')
    digits = re.findall(r'\d', t)
    n = digits[0] if digits else ""

    # --- 1. Supersave Case ---
    if any(k in t for k in ["SUPERSAVE", "SAVE", "ประหยัด"]):
        return "Supersave"

    # --- 2. Buy N Get 1 Case (Thai & English) ---
    if any(k in t for k in ["GET", "แถม"]):
        if n:
            # Special case for 1 Get 1
            if n == "1" or "1แถม1" in t or "1GET1" in t:
                return "Buy 1 Get 1"
            return f"Buy {n} Get 1"
        return "Buy 1 Get"

    # --- 3. Buy N Cheaper Case ---
    if "CHEAPER" in t:
        if n:
            return f"Buy {n} Cheaper"
        return "Buy 2 Cheaper"

    # Fallback: Return raw text if no pattern matches
    return raw_text.strip() if raw_text.strip() else None

# --- START PROCESSING ---

# 2. Get unique badge URLs (Excluding nulls)
unique_urls = [url for url in final_df["badge_url"].unique().to_list() if url and url != "null"]

if not unique_urls:
    print("⚠️ No valid badge URLs found.")
else:
    print(f"🔍 Processing {len(unique_urls)} unique badges...")
    badge_map = {}

    for url in unique_urls:
        try:
            headers = {"User-Agent": "Mozilla/5.0"}
            response = requests.get(url, headers=headers, timeout=10)

            if response.status_code == 200:
                img = Image.open(io.BytesIO(response.content)).convert('RGB')

                # Upscale for OCR accuracy
                img = img.resize((img.width * 4, img.height * 4), resample=Image.LANCZOS)

                # Convert to Numpy Array for EasyOCR
                img_array = np.array(img)

                # Run OCR
                results = reader.readtext(img_array)
                raw_text = " ".join([res[1] for res in results])

                # Process text through our new dynamic function
                label = get_condition_from_text(raw_text)
                badge_map[url] = label
                print(f"✅ {url[-15:]} -> {label}")
            else:
                badge_map[url] = None
        except Exception as e:
            print(f"❌ Error on {url[-15:]}: {e}")
            badge_map[url] = None

    # 3. Apply the results back to the 'condition' column
    final_df = final_df.with_columns(
        pl.col("badge_url").replace(badge_map).alias("condition")
    )

    print("\n✨ Process Complete!")
    # Show results where a condition was found
    display(final_df.filter(pl.col("condition").is_not_null()).head(20))

🔄 Initializing OCR Engine...
🔍 Processing 3 unique badges...


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


✅ 67563483039.png -> Supersave
✅ 56180890355.png -> Buy 1 Get 1
✅ 56180476459.png -> Buy 2 Cheaper

✨ Process Complete!


product_name,sale_price,original_price,condition,badge_url
str,str,str,str,str
"""FINELINE Fabric Softener Sunsh…","""35.00""","""70.00""","""Supersave""","""https://st.bigc-cs.com/cdn-cgi…"
"""FINELINE Fabric Softener Gentl…","""35.00""","""70.00""","""Supersave""","""https://st.bigc-cs.com/cdn-cgi…"
"""FINELINE Fabric Softener Pink …","""35.00""","""70.00""","""Supersave""","""https://st.bigc-cs.com/cdn-cgi…"
"""HYGIENE Expert Wash Concentrat…","""49.00""","""N/A""","""Supersave""","""https://st.bigc-cs.com/cdn-cgi…"
"""FINELINE Fabric Softener Red R…","""35.00""","""70.00""","""Supersave""","""https://st.bigc-cs.com/cdn-cgi…"
…,…,…,…,…
"""FINELINE Fabric Softener Sunsh…","""39.00""","""50.00""","""Supersave""","""https://st.bigc-cs.com/cdn-cgi…"
"""FINELINE Fabric Softener Red R…","""39.00""","""50.00""","""Supersave""","""https://st.bigc-cs.com/cdn-cgi…"
"""HYGIENE Expert Care Concentrat…","""120.00""","""N/A""","""null""","""null"""


In [36]:
final_df.to_pandas()

,product_name,sale_price,original_price,condition,badge_url
0,"FINELINE Fabric Softener Sunshine Gold 1,300 ml.",35.00,70.00,Supersave,https://st.bigc-cs.com/cdn-cgi/image/format=we...
1,"FINELINE Fabric Softener Gentle White 1,300 ml.",35.00,70.00,Supersave,https://st.bigc-cs.com/cdn-cgi/image/format=we...
2,FINELINE Fabric Softener Pink Blossom 1300 ml.,35.00,70.00,Supersave,https://st.bigc-cs.com/cdn-cgi/image/format=we...
3,HYGIENE Expert Wash Concentrate Liquid Deterge...,49.00,N/A,Supersave,https://st.bigc-cs.com/cdn-cgi/image/format=we...
4,FINELINE Fabric Softener Red Romance 1300 ml.,35.00,70.00,Supersave,https://st.bigc-cs.com/cdn-cgi/image/format=we...
...,...,...,...,...,...
327,HYGIENE Perfumed Speed Starch Refill Fresh Oce...,24.00,N/A,null,null
328,HYGIENE Expert Care Concentrated Fabric Soften...,145.00,N/A,null,null
329,ATTACK Lady Elegant Concentrated Powder Laundr...,129.00,174.00,Supersave,https://st.bigc-cs.com/cdn-cgi/image/format=we...
330,ATTACK Charming Romance Concentrated Powder La...,129.00,174.00,Supersave,https://st.bigc-cs.com/cdn-cgi/image/format=we...


In [32]:
# final_df.write_excel(f"big_c_{today_date}.xlsx")

In [5]:
# @title laundry
# เริ่มทำงานและเก็บผลลัพธ์ลงในตัวแปร
df_big_c_laundry = await get_scrape_bigc_multi_pages_polars(base_url="https://www.bigc.co.th/en/category/laundry?brand=187%2C188%2C249%2C186%2C256&limit=100")
# แสดงตัวอย่างข้อมูล
if not df_big_c_laundry.is_empty():
    print(df_big_c_laundry.head(10))

เริ่มการดึงข้อมูลหลายหน้าด้วย Polars...
กำลังดึงข้อมูลหน้า 1: https://www.bigc.co.th/en/category/laundry?brand=187%2C188%2C249%2C186%2C256&limit=100&page=1


[2026-04-28 06:43:56] ERROR: No Cloudflare challenge found.
ERROR:scrapling:No Cloudflare challenge found.
[2026-04-28 06:43:56] INFO: Fetched (200) <GET https://www.bigc.co.th/en/category/laundry?brand=187%2C188%2C249%2C186%2C256&limit=100&page=1> (referer: https://www.google.com/)
INFO:scrapling:Fetched (200) <GET https://www.bigc.co.th/en/category/laundry?brand=187%2C188%2C249%2C186%2C256&limit=100&page=1> (referer: https://www.google.com/)


พบสินค้า 100 รายการในหน้า 1
กำลังดึงข้อมูลหน้า 2: https://www.bigc.co.th/en/category/laundry?brand=187%2C188%2C249%2C186%2C256&limit=100&page=2


[2026-04-28 06:44:06] ERROR: No Cloudflare challenge found.
ERROR:scrapling:No Cloudflare challenge found.
[2026-04-28 06:45:06] INFO: Fetched (200) <GET https://www.bigc.co.th/en/category/laundry?brand=187%2C188%2C249%2C186%2C256&limit=100&page=2> (referer: https://www.google.com/)
INFO:scrapling:Fetched (200) <GET https://www.bigc.co.th/en/category/laundry?brand=187%2C188%2C249%2C186%2C256&limit=100&page=2> (referer: https://www.google.com/)


พบสินค้า 100 รายการในหน้า 2
กำลังดึงข้อมูลหน้า 3: https://www.bigc.co.th/en/category/laundry?brand=187%2C188%2C249%2C186%2C256&limit=100&page=3


...

พบสินค้า 99 รายการในหน้า 3
กำลังดึงข้อมูลหน้า 4: https://www.bigc.co.th/en/category/laundry?brand=187%2C188%2C249%2C186%2C256&limit=100&page=4


[2026-04-28 06:48:32] ERROR: No Cloudflare challenge found.
ERROR:scrapling:No Cloudflare challenge found.
[2026-04-28 06:49:32] INFO: Fetched (200) <GET https://www.bigc.co.th/en/category/laundry?brand=187%2C188%2C249%2C186%2C256&limit=100&page=4> (referer: https://www.google.com/)
INFO:scrapling:Fetched (200) <GET https://www.bigc.co.th/en/category/laundry?brand=187%2C188%2C249%2C186%2C256&limit=100&page=4> (referer: https://www.google.com/)


พบสินค้า 33 รายการในหน้า 4
กำลังดึงข้อมูลหน้า 5: https://www.bigc.co.th/en/category/laundry?brand=187%2C188%2C249%2C186%2C256&limit=100&page=5


[2026-04-28 06:49:43] ERROR: No Cloudflare challenge found.
ERROR:scrapling:No Cloudflare challenge found.
[2026-04-28 06:49:44] INFO: Fetched (200) <GET https://www.bigc.co.th/en/category/laundry?brand=187%2C188%2C249%2C186%2C256&limit=100&page=5> (referer: https://www.google.com/)
INFO:scrapling:Fetched (200) <GET https://www.bigc.co.th/en/category/laundry?brand=187%2C188%2C249%2C186%2C256&limit=100&page=5> (referer: https://www.google.com/)


ไม่พบสินค้าเพิ่มเติมที่หน้า 5 จบการทำงาน

ดึงข้อมูลสำเร็จทั้งหมด 332 รายการ จากหน้า 1 ถึง 4
shape: (10, 5)
┌──────┬─────────────────────────────────┬────────────┬────────────────┬───────────┐
│ page ┆ product_name                    ┆ sale_price ┆ original_price ┆ status    │
│ ---  ┆ ---                             ┆ ---        ┆ ---            ┆ ---       │
│ i64  ┆ str                             ┆ str        ┆ str            ┆ str       │
╞══════╪═════════════════════════════════╪════════════╪════════════════╪═══════════╡
│ 1    ┆ FINELINE Fabric Softener Sunsh… ┆ 35.00      ┆ 70.00          ┆ Available │
│ 1    ┆ FINELINE Fabric Softener Gentl… ┆ 35.00      ┆ 70.00          ┆ Available │
│ 1    ┆ FINELINE Fabric Softener Pink … ┆ 35.00      ┆ 70.00          ┆ Available │
│ 1    ┆ HYGIENE Expert Wash Concentrat… ┆ 49.00      ┆ N/A            ┆ Available │
│ 1    ┆ FINELINE Fabric Softener Red R… ┆ 35.00      ┆ 70.00          ┆ Available │
│ 1    ┆ FINELINE Plus Liquid Laundry D… ┆ 

In [6]:
# @title dishwash
# เริ่มทำงานและเก็บผลลัพธ์ลงในตัวแปร
df_big_c_dishwash = await get_scrape_bigc_multi_pages_polars(base_url="https://www.bigc.co.th/en/category/dishwashing-liquid?limit=100")
# แสดงตัวอย่างข้อมูล
if not df_big_c_dishwash.is_empty():
    print(df_big_c_dishwash.head(10))

เริ่มการดึงข้อมูลหลายหน้าด้วย Polars...
กำลังดึงข้อมูลหน้า 1: https://www.bigc.co.th/en/category/dishwashing-liquid?limit=100&page=1


[2026-04-28 06:50:56] ERROR: No Cloudflare challenge found.
ERROR:scrapling:No Cloudflare challenge found.
[2026-04-28 06:51:56] INFO: Fetched (200) <GET https://www.bigc.co.th/en/category/dishwashing-liquid?limit=100&page=1> (referer: https://www.google.com/)
INFO:scrapling:Fetched (200) <GET https://www.bigc.co.th/en/category/dishwashing-liquid?limit=100&page=1> (referer: https://www.google.com/)


พบสินค้า 89 รายการในหน้า 1
กำลังดึงข้อมูลหน้า 2: https://www.bigc.co.th/en/category/dishwashing-liquid?limit=100&page=2


[2026-04-28 06:52:06] ERROR: No Cloudflare challenge found.
ERROR:scrapling:No Cloudflare challenge found.
[2026-04-28 06:52:07] INFO: Fetched (200) <GET https://www.bigc.co.th/en/category/dishwashing-liquid?limit=100&page=2> (referer: https://www.google.com/)
INFO:scrapling:Fetched (200) <GET https://www.bigc.co.th/en/category/dishwashing-liquid?limit=100&page=2> (referer: https://www.google.com/)


พบสินค้า 2 รายการในหน้า 2
กำลังดึงข้อมูลหน้า 3: https://www.bigc.co.th/en/category/dishwashing-liquid?limit=100&page=3


[2026-04-28 06:52:17] ERROR: No Cloudflare challenge found.
ERROR:scrapling:No Cloudflare challenge found.
[2026-04-28 06:52:17] INFO: Fetched (200) <GET https://www.bigc.co.th/en/category/dishwashing-liquid?limit=100&page=3> (referer: https://www.google.com/)
INFO:scrapling:Fetched (200) <GET https://www.bigc.co.th/en/category/dishwashing-liquid?limit=100&page=3> (referer: https://www.google.com/)


ไม่พบสินค้าเพิ่มเติมที่หน้า 3 จบการทำงาน

ดึงข้อมูลสำเร็จทั้งหมด 91 รายการ จากหน้า 1 ถึง 2
shape: (10, 5)
┌──────┬─────────────────────────────────┬────────────┬────────────────┬───────────┐
│ page ┆ product_name                    ┆ sale_price ┆ original_price ┆ status    │
│ ---  ┆ ---                             ┆ ---        ┆ ---            ┆ ---       │
│ i64  ┆ str                             ┆ str        ┆ str            ┆ str       │
╞══════╪═════════════════════════════════╪════════════╪════════════════╪═══════════╡
│ 1    ┆ SUNLIGHT Lemon Turbo Dishwashi… ┆ 88.00      ┆ 130.00         ┆ Available │
│ 1    ┆ PRO Dishwashing liquid Intense… ┆ 10.50      ┆ N/A            ┆ Available │
│ 1    ┆ BIG C HAPPY PRICE Dishwashing … ┆ 25.00      ┆ N/A            ┆ Available │
│ 1    ┆ SUNLIGHT Lemon Turbo Dishwashi… ┆ 159.00     ┆ 179.00         ┆ Available │
│ 1    ┆ SUNLIGHT Lemon Turbo Dishwashi… ┆ 29.00      ┆ 32.00          ┆ Available │
│ 1    ┆ PRO Dishwashing liquid Tropica… ┆ 1

In [37]:
df_big_c = final_df.clone()

In [39]:
df_big_c_sel = df_big_c.select([
    pl.col("product_name").alias("name"),
    # strict=False จะเปลี่ยนทุกอย่างที่แปลงเป็น Float ไม่ได้ให้เป็น Null
    pl.col("sale_price").cast(pl.Float64, strict=False).alias("promotion_price"),
    pl.col("original_price").cast(pl.Float64, strict=False).alias("original_price"),
    pl.col("condition")
])

In [40]:
# @title udf data-prep
def re_evaluate_price(df: pl.DataFrame) -> pl.DataFrame:
    """
    Standardizes pricing logic:
    1. If original_price is missing, move the promotion_price to it.
    2. If promotion_price matches the original_price, set it to Null.
    """
    return (
        df.with_columns(
            # Step 1: Fix missing original prices by 'swapping' from promotion_price
            pl.when(pl.col("original_price").is_null() & pl.col("promotion_price").is_not_null())
            .then(pl.col("promotion_price"))
            .otherwise(pl.col("original_price"))
            .alias("original_price")
        )
        .with_columns(
            # Step 2: Now that original_price is populated, nullify redundant promotions
            pl.when(pl.col("promotion_price") == pl.col("original_price"))
            .then(None)
            .otherwise(pl.col("promotion_price"))
            .alias("promotion_price")
        )
    )

# How to use it:
df_prep_big_c = re_evaluate_price(df_big_c_sel)

In [41]:
# @title udf Transform (Fixed Volume Extraction)
def parse_product_names(df: pl.DataFrame, shop_name: str) -> pl.DataFrame:
    """
    Pass any supermarket dataframe through this function to standardize the columns.
    Fixed to handle thousands separators (e.g., 1,300 ml).
    """
    # 1. Setup the dynamic date
    today_date = date.today().strftime("%Y-%m-%d")

    # 2. Updated patterns
    # Added [\d,.]+ to capture digits, commas, and dots
    quant_unit_pattern = r"(?i)([\d,.]+)\s*(ML|G|KG|L)\b"
    pack_pattern = r"(?i)(PACK\s*\d*\s*FREE\s*\d+|PACK\s*\d*\s*\+\s*\d+|PACK\s*\d+|TWINPACK|\bX\s*\d+\b|P?\d+\s*\+\s*\d+|\(\d+\+\d+\)|\d+\s*FREE\s*\d+|\bPACK\b)"

    # 3. Apply the Polars transformation
    return df.with_columns(
        pl.lit(today_date).alias("Date"),

        # Extract Brand
        pl.col("name").str.split(" ").list.first().alias("Brand"),

        # Fixed Volume Extraction:
        # 1. Extract the group (e.g., "1,300")
        # 2. Replace commas with nothing so it becomes "1300"
        # 3. Cast to Integer
        pl.col("name")
          .str.extract(quant_unit_pattern, 1)
          .str.replace_all(",", "")
          .cast(pl.Int64, strict=False)
          .alias("Volume"),

        # Extract Unit
        pl.col("name").str.extract(quant_unit_pattern, 2).str.to_uppercase().alias("Unit"),

        # Extract Pack size
        pl.col("name").str.extract(pack_pattern, 1).str.to_uppercase().alias("Pack"),

        # Add the dynamic Shop identifier
        pl.lit(shop_name).alias("Retailer")
    )

In [42]:
df_trans_big_c = parse_product_names(df_prep_big_c, "BigC")

In [43]:
df_trans_big_c

name,promotion_price,original_price,condition,Date,Brand,Volume,Unit,Pack,Retailer
str,f64,f64,str,str,str,i64,str,str,str
"""FINELINE Fabric Softener Sunsh…",35.0,70.0,"""Supersave""","""2026-04-28""","""FINELINE""",1300,"""ML""",null,"""BigC"""
"""FINELINE Fabric Softener Gentl…",35.0,70.0,"""Supersave""","""2026-04-28""","""FINELINE""",1300,"""ML""",null,"""BigC"""
"""FINELINE Fabric Softener Pink …",35.0,70.0,"""Supersave""","""2026-04-28""","""FINELINE""",1300,"""ML""",null,"""BigC"""
"""HYGIENE Expert Wash Concentrat…",null,49.0,"""Supersave""","""2026-04-28""","""HYGIENE""",600,"""ML""",null,"""BigC"""
"""FINELINE Fabric Softener Red R…",35.0,70.0,"""Supersave""","""2026-04-28""","""FINELINE""",1300,"""ML""",null,"""BigC"""
…,…,…,…,…,…,…,…,…,…
"""HYGIENE Perfumed Speed Starch …",null,24.0,"""null""","""2026-04-28""","""HYGIENE""",900,"""ML""",null,"""BigC"""
"""HYGIENE Expert Care Concentrat…",null,145.0,"""null""","""2026-04-28""","""HYGIENE""",1100,"""ML""",null,"""BigC"""
"""ATTACK Lady Elegant Concentrat…",129.0,174.0,"""Supersave""","""2026-04-28""","""ATTACK""",1600,"""G""",null,"""BigC"""


In [44]:
df_trans_big_c.write_excel(f"big_c_result_{today_date}.xlsx")